# **COMPETITIVE PRICING ANALYSIS FOR PERSONAL CARE E-COMMERCE**
##### **by Lucila Aldana Quiñonez | Marketing Data Analyst**

In [14]:
# Import libraries

import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, date
import time

# **Competitor A: GPSFarma**

In [168]:
# Configuración

category_urls_a = [
    "https://gpsfarma.com/categorias/cuidado-de-la-piel/corporales.html",
    "https://gpsfarma.com/categorias/cuidado-de-la-piel/faciales.html",
    "https://gpsfarma.com/categorias/cuidado-de-la-piel/solares.html",
    "https://gpsfarma.com/categorias/cuidado-oral.html",
    "https://gpsfarma.com/categorias/cuidado-del-cabello.html"
]

competitor_a = "GPSFarma"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [169]:
# Limpieza de precios

def clean_price(price_text):
    price_text = (
        price_text.replace("$", "")
        .replace(".", "")
        .replace(",", ".")
        .replace("\xa0", "")
        .strip()
    )
    try:
        return float(price_text)
    except ValueError:
        return None

In [170]:
# Extracción de productos

def extract_product_data(product, category_url):
    name_tag = product.find("strong", class_="product-item-name")
    brand_tag = product.find("div", class_="product-item-brand")
    price_tag = product.find("span", class_="price")
    link_tag = product.find("a", class_="product-item-link")
    discount_tag = product.find("img", alt=lambda x: x and "Off" in x)

    raw_name = name_tag.get_text(" ", strip=True) if name_tag else None

    # filtro productos basura
    if not raw_name or not link_tag:
        return None

    availability = "Sin stock" if "Sin stock" in product.get_text() else "En stock"

    return {
        "product_name": raw_name,
        "brand": brand_tag.get_text(strip=True) if brand_tag else None,
        "price": clean_price(price_tag.get_text()) if price_tag else None,
        "discount": discount_tag.get("alt") if discount_tag else None,
        "availability": availability,
        "category": category_url,
        "subcategory": category_url.split("/")[-1].replace(".html", ""),
        "product_url": link_tag.get("href"),
        "competitor": competitor_a,
        "scraping_date": datetime.today().date()
    }

In [171]:
# Scraping por categoría

def scrape_gpsfarma_category(category_url, max_pages=150):
    page = 1
    products = []
    seen_urls = set()

    while page <= max_pages:
        url = f"{category_url}?p={page}"
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        product_cards = soup.find_all("li", class_="product-item")

        if not product_cards:
            break

        print(f"Scrapeando página {page} - productos: {len(product_cards)}")

        new_products = 0

        for product in product_cards:
            product_data = extract_product_data(product, category_url)
            if not product_data:
                continue

            if product_data["product_url"] in seen_urls:
                continue

            seen_urls.add(product_data["product_url"])
            products.append(product_data)
            new_products += 1

        if new_products == 0:
            print("No hay productos nuevos, se detiene el scraping.")
            break

        page += 1

    return products

In [172]:
# Scraping total GPSFarma

all_products = []

for url in category_urls_a:
    print(f"\nCategoría: {url}")
    products = scrape_gpsfarma_category(url, max_pages=150)
    all_products.extend(products)

print(f"\nTotal productos extraídos: {len(all_products)}")


Categoría: https://gpsfarma.com/categorias/cuidado-de-la-piel/corporales.html
Scrapeando página 1 - productos: 13
Scrapeando página 2 - productos: 13
Scrapeando página 3 - productos: 13
Scrapeando página 4 - productos: 13
Scrapeando página 5 - productos: 13
Scrapeando página 6 - productos: 13
Scrapeando página 7 - productos: 13
Scrapeando página 8 - productos: 13
Scrapeando página 9 - productos: 13
Scrapeando página 10 - productos: 13
Scrapeando página 11 - productos: 13
Scrapeando página 12 - productos: 13
Scrapeando página 13 - productos: 13
Scrapeando página 14 - productos: 13
Scrapeando página 15 - productos: 13
Scrapeando página 16 - productos: 13
Scrapeando página 17 - productos: 13
No hay productos nuevos, se detiene el scraping.

Categoría: https://gpsfarma.com/categorias/cuidado-de-la-piel/faciales.html
Scrapeando página 1 - productos: 13
Scrapeando página 2 - productos: 13
Scrapeando página 3 - productos: 13
Scrapeando página 4 - productos: 13
Scrapeando página 5 - productos

In [174]:
# Guardar CSV

today = date.today().strftime("%Y-%m-%d")
df = pd.DataFrame(all_products)

df.to_csv(
    f"products_gpsfarma_{today}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo CSV guardado correctamente.")

Archivo CSV guardado correctamente.


# **Competitor B: Farmaonline**

In [2]:
# Configuración

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

BASE_URL = "https://www.farmaonline.com"
competitor_b = "Farmaonline"

In [3]:
# Limpieza de precios

def clean_price(price_text):
    price_text = (
        price_text.replace("$", "")
        .replace(".", "")
        .replace(",", ".")
        .replace("\xa0", "")
        .strip()
    )
    try:
        return float(price_text)
    except:
        return None

In [4]:
# Extracción de productos

def scrape_farmaonline_first_page(url, category, subcategory=None):
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    product_cards = soup.find_all("article")

    products = []

    for product in product_cards:
        name_tag = product.find(
            "span",
            class_="vtex-product-summary-2-x-productBrand vtex-product-summary-2-x-brandName"
        )
        link_tag = product.find("a", href=True)

        if not name_tag or not link_tag:
            continue

        brand_tag = product.find("span", class_="vtex-product-summary-2-x-productBrandName")
        price_tag = product.find("span", class_="vtex-product-price-1-x-sellingPriceValue")
        discount_tag = product.find("span", class_="vtex-product-price-1-x-savingsPercentage")

        availability = (
            "Sin stock"
            if product.find("div", class_="farmaonline-product-summmary-elements-0-x-nope_stock")
            else "En stock"
        )

        products.append({
            "product_name": name_tag.get_text(strip=True),
            "brand": brand_tag.get_text(strip=True) if brand_tag else None,
            "price": clean_price(price_tag.get_text()) if price_tag else None,
            "discount": discount_tag.get_text(strip=True) if discount_tag else None,
            "availability": availability,
            "category": category,
            "subcategory": subcategory,
            "product_url": BASE_URL + link_tag["href"],
            "competitor": competitor_b,
            "scraping_date": datetime.today().date()
        })

    return products

In [6]:
url = "https://www.farmaonline.com/cuidado-personal/cuidado-de-la-piel/corporales?order=OrderByTopSaleDESC"

data = scrape_farmaonline_first_page(
    url=url,
    category="cuidado-de-la-piel",
    subcategory="corporales"
)

print(f"Productos extraídos: {len(data)}")

Productos extraídos: 0


#### Farmaonline utiliza renderizado dinámico mediante JavaScript (VTEX), lo que impide la extracción de productos mediante scraping HTML tradicional con Requests y BeautifulSoup.

# <span style="color:red">**Competitor C: Farmacity**</span>

In [43]:
# Configuración

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

base_url = "https://www.farmacity.com"
COMPETITOR = "Farmacity"

categories = [
    {
        "category": "dermocosmetica",
        "subcategory": "corporal",
        "url": "https://www.farmacity.com/dermocosmetica/corporal"
    },
    {
        "category": "dermocosmetica",
        "subcategory": "rostro",
        "url": "https://www.farmacity.com/dermocosmetica/rostro"
    },
    {
        "category": "dermocosmetica",
        "subcategory": "solar",
        "url": "https://www.farmacity.com/dermocosmetica/solar"
    },
    {
        "category": "cuidado-personal",
        "subcategory": "cuidado-oral",
        "url": "https://www.farmacity.com/cuidado-personal/cuidado-oral"
    },
    {
        "category": "dermocosmetica",
        "subcategory": "pelo",
        "url": "https://www.farmacity.com/dermocosmetica/pelo"
    }
]

In [44]:
# Limpieza de precios

def clean_price(price_text):
    if not price_text:
        return None
    price_text = price_text.replace("$", "").replace(".", "").replace(",", ".")
    return float(price_text)

In [45]:
# Extracción de productos

def extract_product_data_farmacity(product, category, subcategory):
    # Nombre
    name_tag = product.select_one(
        "span.vtex-product-summary-2-x-productBrand"
    )

    # Marca
    brand_tag = product.select_one(
        "span.vtex-product-summary-2-x-productBrandName"
    )

    # Precio (contenedor correcto)
    price_container = product.select_one(
        "span.vtex-product-price-1-x-sellingPriceValue"
    )

    # Descuento
    discount_tag = product.select_one(
        "p.farmacityar-app-highlights-admin-0-x-discountText"
    )

    # Link
    link_tag = product.find("a", href=True)

    if not name_tag or not link_tag:
        return None

    # Limpieza de precio
    price = None
    if price_container:
        price_text = (
            price_container.get_text(strip=True)
            .replace("$", "")
            .replace(".", "")
            .replace(",", ".")
        )
        try:
            price = float(price_text)
        except ValueError:
            price = None

    return {
        "product_name": name_tag.get_text(strip=True),
        "brand": brand_tag.get_text(strip=True) if brand_tag else None,
        "price": price,
        "discount": discount_tag.get_text(strip=True) if discount_tag else None,
        "availability": "En stock",
        "category": category,
        "subcategory": subcategory,
        "product_url": BASE_URL + link_tag["href"],
        "competitor": COMPETITOR,
        "scraping_date": datetime.today().date()
    }

In [46]:
def scrape_farmacity_category(url, category, subcategory):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.select("div[data-af-element='search-result']")

    results = []

    for product in products:
        data = extract_product_data_farmacity(product, category, subcategory)
        if data:
            results.append(data)

    print(f"{subcategory}: {len(results)} productos extraídos")
    return results


In [47]:
all_products = []

for cat in categories:
    products = scrape_farmacity_category(
        url=cat["url"],
        category=cat["category"],
        subcategory=cat["subcategory"]
    )
    all_products.extend(products)

corporal: 7 productos extraídos
rostro: 7 productos extraídos
solar: 7 productos extraídos
cuidado-oral: 7 productos extraídos
pelo: 7 productos extraídos


In [48]:
# Guardar CSV

today = date.today().strftime("%Y-%m-%d")
df = pd.DataFrame(all_products)

df.to_csv(
    f"products_farmacity_{today}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\nTotal productos Farmacity: {len(df)}")
print("Archivo CSV guardado correctamente.")


Total productos Farmacity: 35
Archivo CSV guardado correctamente.


#### Farmacity implementa carga dinámica de productos mediante infinite scroll (VTEX). Este script extrae únicamente los productos visibles en la primera carga HTML, garantizando estabilidad y reproducibilidad del scraping.

# <span style="color:red">**Competitor D: Farmalife**</span>